# Atera example workflow (R / Seurat)

This notebook replicates, in R using Seurat's Atera reader, the workflow from the
Python reference notebook at
`/mnt/home/stephen.williams/Apps/spatialdata-io/notebooks/atera_example.ipynb`, which
uses `spatialdata_io.atera()` + scanpy to load an Atera output bundle, inspect it,
make QC/spatial plots, and run a normalize → HVG → PCA → cluster → UMAP pipeline.

**Kernel:** select the `R 4.4.3 (Conda)` kernel.

**Data:** a tiny dev-scale bundle (13 cells, 53 real genes + a handful of
control/codeword features, immune-marker panel) is used here so the whole notebook
runs in seconds. Swap `BUNDLE_PATH` below for a full-size bundle to see a realistic
clustering/UMAP result; see `atera_exploration.ipynb` for that at scale.

**BPCells by default:** `LoadAtera()` loads its counts matrices lazily, on-disk, via
BPCells by default — even for a bundle this tiny — so `atera.obj` below is
`IterableMatrix`-backed rather than a plain in-memory `dgCMatrix`, exactly as it would be
for a full-size bundle. This is intentional: write your analysis code assuming the
BPCells path, since that's what it'll actually be running against in practice. Pass
`bpcells.dir = FALSE` to `LoadAtera()`/`ReadAtera()` if you want to opt out and always get
an in-memory `dgCMatrix` instead (e.g. for a quick one-off look you don't want cached to
disk).

**Two capability gaps vs. the Python reader**, called out inline below rather than
worked around:

1. **No morphology image loading.** `spatialdata_io.atera()` loads the multiscale DAPI
   morphology image; `ReadAtera`/`LoadAtera` currently only expose
   `outs = c("matrix", "centroids", "segmentations", "nucleus_segmentations", "microns")` —
   there is no `"images"` option. This notebook has no equivalent of the Python
   notebook's raw image QC panel.
2. **No codeword-level transcript resolution.** Molecules are resolved to **gene**
   identity only (via `transcripts.zarr.zip`'s `gene_names`), with no codeword column
   exposed, so the Python notebook's `by_codeword=True` per-codeword transcript
   breakdown isn't reproducible here.

See the end of this notebook for a pointer to `atera_exploration.ipynb`, which covers
customizing the BPCells cache (persistent directories, feature-type scoping) and the
sketch-based workflow for scaling to very large cell counts — not repeated here.

In [ ]:
suppressPackageStartupMessages({
  library(devtools)
  library(ggplot2)
})

# Load the local, editable copy of Seurat (with ReadAtera/LoadAtera) instead of an
# installed version, so changes under R/ are picked up without reinstalling.
devtools::load_all("/mnt/home/stephen.williams/Apps/seurat", quiet = TRUE)

# Tiny dev-scale bundle: 13 cells, 132 total features (53 Gene Expression + control/
# codeword types), immune-marker panel.
BUNDLE_PATH <- "/mnt/deck/2/stephen.williams/atera_dev/tiny_atera_dataset"

## Load data

The Python notebook builds a `SpatialData` object (Images + Points + Shapes + Tables)
via `spatialdata_io.atera()`. The Seurat equivalent is a single `Seurat` object built
by `LoadAtera()`: the Gene Expression matrix becomes the primary assay, the control/
codeword feature types become secondary assays, cell centroids + segmentation polygons
become an `FOV`, and (since `molecule.coordinates = TRUE`) transcript coordinates are
preloaded lazily for later on-demand gene fetches via `LoadAteraMolecules()`. As noted
above, the counts matrices are also BPCells-backed by default (no extra arguments
needed).

In [ ]:
print(system.time(
  atera.obj <- LoadAtera(
    data.dir = BUNDLE_PATH,
    fov = "fov",
    assay = "Atera",
    cell.centroids = TRUE,
    molecule.coordinates = TRUE,
    segmentations = "cell"
  )
))
atera.obj

# BPCells-backed by default, even for this tiny bundle -- pass bpcells.dir = FALSE to
# LoadAtera() above to opt out and always get a plain in-memory dgCMatrix instead.
class(atera.obj[["Atera"]]$counts)

## Inspect data structures

Analogous to printing/inspecting the Python `SpatialData` object's Images/Points/Shapes/Tables.

In [ ]:
cat("Assays:", paste(Assays(atera.obj), collapse = ", "), "\n")
cat("FOVs:", paste(Images(atera.obj), collapse = ", "), "\n")
head(atera.obj[[]])

## QC

The Python notebook computes per-cell transcript counts and gene counts by manually
iterating dask blocks of the backing array (`sc.pp.calculate_qc_metrics` doesn't work
directly on the lazy matrix). Seurat computes `nCount_Atera`/`nFeature_Atera`
automatically at object-creation time, so no manual block iteration is needed here.

In [ ]:
VlnPlot(atera.obj, features = c("nCount_Atera", "nFeature_Atera"), pt.size = 0, combine = TRUE)

In [ ]:
ImageFeaturePlot(atera.obj, fov = "fov", features = "nCount_Atera", size = 1)

In [ ]:
ImageFeaturePlot(atera.obj, fov = "fov", features = "nFeature_Atera", size = 1)

## Single-gene transcript plot

Mirrors the Python notebook's `read_transcripts_for_genes` + spatial scatter for one
gene. `LoadAteraMolecules()` fetches a gene's transcript coordinates on demand (only
that gene is decoded from `transcripts.zarr.zip`) and attaches them to the FOV.

In [ ]:
print(system.time(atera.obj <- LoadAteraMolecules(atera.obj, genes = "KLRD1")))
ImageDimPlot(
  atera.obj, fov = "fov", size = 1, cols = "grey80",
  molecules = "KLRD1", mols.size = 1, mols.cols = "orange"
)

## Gene expression per cell

Mirrors the Python notebook's `read_table_for_genes` + spatial expression plot.

In [ ]:
ImageFeaturePlot(atera.obj, fov = "fov", features = "KLRD1", size = 1)

## Crop to a region

Mirrors the Python notebook's region-cropping cell. `Crop()` subsets an `FOV`'s
centroids/segmentations to a bounding box; `coords = "tissue"` matches x/y directly to
the underlying data (micron) coordinates. (Bounds below are sized for this bundle's
small ~25x25 micron extent - adjust for a full-size bundle. Note `coords = "plot"`,
the other option, swaps the x/y axes to match image-plot pixel space instead - use
`"tissue"` when thinking in data coordinates as done here.)

In [ ]:
atera.obj[["zoom"]] <- Crop(atera.obj[["fov"]], x = c(10, 25), y = c(5, 20), coords = "tissue")
ImageFeaturePlot(atera.obj, fov = "zoom", features = "KLRD1", size = 2)

## Multi-gene transcript plot

Matches the Python notebook's gene pair exactly - both KLRD1 and TOX3 are present in
this panel.

In [ ]:
genes <- c("KLRD1", "TOX3")
print(system.time(atera.obj <- LoadAteraMolecules(atera.obj, genes = genes)))
ImageDimPlot(
  atera.obj, fov = "fov", size = 1, cols = "grey80",
  molecules = genes, mols.size = 1, mols.cols = c("orange", "cyan")
)

> **Capability gap:** the Python notebook can also break these transcript plots down
> `by_codeword=True` (deprecated/blank/control codewords vs. real gene calls). The R
> reader's `LoadAteraMolecules()` resolves transcripts to gene identity only, with no
> codeword column exposed, so that per-codeword breakdown isn't reproducible here.

## Cell boundaries

Mirrors the Python notebook's boundary/shapes plot.

In [ ]:
ImageDimPlot(atera.obj, fov = "fov", boundaries = "segmentations", size = 1, cols = "grey60")

## Normalize, HVG, PCA, clustering, UMAP

The Python notebook can't run `sc.pp.highly_variable_genes`/`sc.pp.pca` directly on
the dask-backed full matrix (it blows up memory), so it hand-rolls a streaming
mean/variance pass (`streaming_gene_mean_var`) and an `IncrementalPCA` fit in
mini-batches, then runs `sc.pp.neighbors`/`sc.tl.leiden` on the resulting embedding.

In Seurat, `NormalizeData`, `FindVariableFeatures`, `ScaleData`, `RunPCA`,
`FindNeighbors`, and `FindMarkers` already dispatch natively on `IterableMatrix`
(BPCells) input (see `atera_exploration.ipynb`'s BPCells section) - so the same
memory-safety Python achieves with hand-written streaming code comes for free here with
the standard Seurat calls below. `atera.obj` is BPCells-backed by default (confirmed
above), so this is genuinely exercising the `IterableMatrix` code path, not just the
in-memory one, even at this tiny scale; `npcs`/`dims` are capped at 10 since there are
only 13 cells here - use the usual 30-50 for a full-size bundle.

In [ ]:
print(system.time(atera.obj <- NormalizeData(atera.obj, verbose = FALSE)))
print(system.time(atera.obj <- FindVariableFeatures(atera.obj, verbose = FALSE)))
print(system.time(atera.obj <- ScaleData(atera.obj, verbose = FALSE)))
print(system.time(atera.obj <- RunPCA(atera.obj, npcs = 10, verbose = FALSE)))
print(system.time(atera.obj <- FindNeighbors(atera.obj, dims = 1:10, verbose = FALSE)))
print(system.time(atera.obj <- FindClusters(atera.obj, resolution = 1.0, verbose = FALSE)))
print(system.time(atera.obj <- RunUMAP(atera.obj, dims = 1:10, n.neighbors = 5, verbose = FALSE)))
cat("num clusters:", length(unique(atera.obj$seurat_clusters)), "\n")

In [ ]:
DimPlot(atera.obj, group.by = "seurat_clusters", label = TRUE) + NoLegend()

## Clusters over tissue

Mirrors the Python notebook's final spatial Leiden-cluster plot.

In [ ]:
ImageDimPlot(atera.obj, fov = "fov", group.by = "seurat_clusters", size = 1)

## Next steps

This bundle is intentionally tiny (13 cells) so the notebook runs end-to-end in
seconds; the cluster/UMAP results above are for illustration only, not biologically
meaningful. For Atera's real whole-transcriptome bundles (hundreds of thousands to
millions of cells), see `atera_exploration.ipynb` for:

- **Customizing the BPCells cache.** `bpcells.dir` is already on by default above
  (session-scoped, disappears with the R session); pointing it at an explicit, persistent
  directory instead makes the cache survive across sessions, and `feature.types` further
  restricts decoding/caching to just the feature type(s) you need.
- **Sketch-based analysis** (`SketchData()` → cluster/UMAP on a leverage-score subsample
  → `ProjectData()` back onto the full dataset) for datasets too large to cluster
  directly in reasonable time.